# **LLM Fine-Tuning**

### __Objective:__

In this demo, you will fine-tune the Falcon RW 1B using Parameter-Efficient Fine-Tuning (PEFT) with LoRA.
You will tokenize a subset of WikiText-2 and configure key LoRA parameters (rank, scaling, dropout) for efficient training.
Finally, compare the model outputs before and after fine-tuning to showcase the method's effectiveness.

---

### **Note:**  
- Before running any demo, ensure that the **requirements.txt** file is installed. This file contains all the required dependencies for **all demos and guided practices under Building LLM Applications**.
- If the dependencies were already installed earlier (after creating the virtual environment), there is no need to install them again. You can directly proceed with running the demo.
- Refer to Lesson_01 **Demo_01_Zero_Shot_Prompting.ipynb** Step 1 for creating a virtual environment and installing the requirements.txt 
- Ensure you select the right kernel **Python (myenv)** while running the demos
---


### **Steps to be followed:**

1. Install required packages and import libraries
2. Set device and load pre-trained model
3. Configure PEFT with LoRA
4. Move the model to the device
5. Load and preprocess the dataset
6. Define a custom data collator
7. Generate and store model outputs before fine-tuning
8. Configure training arguments and fine-tune the model
9. Compare model outputs after fine-tuning

---

### **Step 1: Install required packages and import libraries**

- Install and import essential libraries for model loading, dataset management, and fine-tuning

Installing and importing packages like transformers, peft, and datasets prepares the environment for loading pre-trained models, applying parameter-efficient fine-tuning, and managing datasets. This sets the foundation for the entire demo.

In [8]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [9]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    default_data_collator
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

### **Step 2: Set device and load pre-trained model**

- Determine if a GPU is available and load the pre-trained Falcon RW 1B model along with its tokenizer

Using a GPU (if available) significantly accelerates training. Loading a pre-trained model like Falcon RW 1B gives us a base model that we can fine-tune, saving both time and resources compared to training from scratch.

In [10]:
# Set device to GPU if available (e.g., T4), otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "tiiuae/falcon-rw-1b"  # ✅ use the lighter version for stability

# Quantization config — uses much less VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # 4-bit loading (big memory saver)
    bnb_4bit_compute_dtype=torch.float16,  # compute safely in fp16
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Load model with automatic layer distribution
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",  # auto-distribute GPU/CPU layers safely
    trust_remote_code=True  # Falcon requires this for now
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model.eval()

print(f"✅ Falcon RW 1B loaded successfully on: {model.device}")


✅ Falcon 1B loaded successfully on: cuda:0


In [12]:
print(model)

FalconForCausalLM(
  (transformer): FalconModel(
    (word_embeddings): Embedding(50304, 2048)
    (h): ModuleList(
      (0-23): 24 x FalconDecoderLayer(
        (self_attention): FalconAttention(
          (query_key_value): Linear4bit(in_features=2048, out_features=6144, bias=True)
          (dense): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (mlp): FalconMLP(
          (dense_h_to_4h): Linear4bit(in_features=2048, out_features=8192, bias=True)
          (act): GELU(approximate='none')
          (dense_4h_to_h): Linear4bit(in_features=8192, out_features=2048, bias=True)
        )
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      )
    )
    (ln_f): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=2048, out_features=50304,

### **Step 3: Configure PEFT with LoRA**

- We now set up the Low-Rank Adaptation (LoRA) configuration for PEFT. This involves defining several parameters:

    - `r=8`: This is the LoRA rank. It determines the size of the low-rank matrices added to the model. A higher rank can capture more nuances but increases parameters slightly.
    - `lora_alpha=32`: This scaling factor adjusts the magnitude of the LoRA weights. It helps in stabilizing training by scaling the low-rank updates.
    - `target_modules=["q_proj", "v_proj"]`: Specifies which layers to apply LoRA to. For Falcon 1B, targeting the *q_proj and v_proj* module (the attention projection layer) is common since these layers significantly impact the model's performance.
    - `lora_dropout=0.1`: This dropout rate is used on the LoRA layers to regularize the training and prevent overfitting.
    - `bias="none"`: Indicates that the bias parameters are not being fine-tuned.
    - `task_type="CAUSAL_LM"`: Specifies that our task is causal language modeling.


This step is crucial because it configures the PEFT method. By adding trainable low-rank matrices only to key components, we significantly reduce the number of parameters to update, making fine-tuning both memory- and compute-efficient.

In [15]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,                      # smaller rank for low memory
    lora_alpha=16,
    target_modules=["query_key_value", "dense"],  # ✅ Falcon-specific layers
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)


The Falcon 1B model is now wrapped with LoRA-based PEFT. Only the additional low-rank parameters in the targeted modules will be updated during fine-tuning, making the process more efficient while retaining performance.

### **Step 4: Move the model to the device**

- Move the model to the GPU (if available) to ensure faster training and inference


Transferring the model to the correct device (GPU/CPU) is essential for performance. GPUs, in particular, accelerate the matrix operations involved in training deep neural networks.

In [16]:
# Move the model to the chosen device (GPU)
model = model.to(device)

### **Step 5: Load and preprocess the dataset**

- Load a subset (70%) of the WikiText-2 dataset, which contains raw text data
- Tokenize the text using the Falcon 1B tokenizer with a maximum sequence length of 128 tokens
- Filter out any examples that yield empty token sequences


Preprocessing the data is a critical step before training. Tokenizing converts raw text into numerical tokens that the model can understand. Filtering ensures that only valid data is passed to the model, which improves training quality.

In [17]:
# Load a subset of WikiText-2 for demonstration
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:70%]")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) > 0)

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/25703 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25703 [00:00<?, ? examples/s]

### **Step 6: Define a custom data collator**

- Define a custom data collator function to prepare batches of data for training. This function ensures:
    - The `input_ids` are cast to long tensors.
    - The `labels` are set correctly. If labels aren’t provided, they are set to be the same as the input_ids.


A data collator is used to combine individual examples into a batch. This ensures that all sequences in the batch have consistent formatting, which is critical for training stability and performance.

In [21]:
def collate_fn(features):
    batch = default_data_collator(features)
    batch["input_ids"] = batch["input_ids"].long()
    batch["labels"] = batch["input_ids"].clone()
    return batch


### **Step 7: Generate and store model outputs before fine-tuning**
- Generate outputs for several predefined prompts
- These outputs represent the baseline performance of the model in its pre-fine-tuned state and are stored for later comparison.

Capturing the model’s behavior before fine-tuning is crucial for demonstrating how PEFT changes the model's responses. This baseline is essential for a clear before-and-after comparison.

In [22]:

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

prompts = [
    "What is artificial intelligence?",
    "Explain the concept of quantum computing in simple terms.",
    "Describe the impact of renewable energy on the environment."
]

print("\n=== Generating outputs BEFORE fine-tuning ===")
before_outputs = {}
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            use_cache=False,        # ✅ prevents Falcon cache bug
            max_new_tokens=64,      # ✅ shorter output for low VRAM
            temperature=0.8,
            top_p=0.9,
        )

    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    before_outputs[prompt] = text
    print(f"\nPrompt: {prompt}\nBefore: {text}\n{'-'*60}")



=== Generating outputs BEFORE fine-tuning ===

Prompt: What is artificial intelligence?
Before: What is artificial intelligence?
Artificial intelligence (AI) is a field of study that aims to create machines that think and act like humans.
Artificial intelligence is a field of study that aims to create machines that think and act like humans.
Artificial intelligence is a field of study that aims to create machines that think and act like
------------------------------------------------------------

Prompt: Explain the concept of quantum computing in simple terms.
Before: Explain the concept of quantum computing in simple terms.
Quantum computing is a new way of computing that uses quantum mechanics to solve problems that are too large for conventional computers.
What is quantum computing?
Quantum computing is a new way of computing that uses quantum mechanics to solve problems that are too large for conventional computers.
What is quantum computing?
Quant
------------------------------

### **Step 8: Configure training arguments and fine-tune the model**

We now set up the training configuration with specific parameters:

- `output_dir`: Directory to save the fine-tuned model
- `run_name`: Name for the training run
- `max_steps`: Limits the number of training steps (set to 50 for this demo)
- `per_device_train_batch_size`: Batch size for each device
- `learning_rate`: The learning rate for fine-tuning
- `logging_steps and save_steps`: Frequency of logging and saving checkpoints
- `num_train_epochs`: Number of training epochs
- `fp16`: Enables mixed precision training for speed on GPUs
- `no_cuda`: Indicates that CUDA (GPU) should be used if available
- `report_to`: Disables external logging tools like Weights & Biases

Then we initialize the Hugging Face Trainer with our model, training arguments, dataset, and data collator, and run the training process.

This step fine-tunes the model using our dataset and LoRA configuration. The training arguments control the fine-tuning process, and running the trainer updates only the low-rank parameters introduced by LoRA.

In [28]:

print("\n⏳ Fine-tuning Falcon-RW-1B with LoRA...")

training_args = TrainingArguments(
    output_dir="./falcon_rw1b_lora_finetuned",
    overwrite_output_dir=True,
    max_steps=50,                        # ✅ small training steps
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=20,
    num_train_epochs=1,
    fp16=torch.cuda.is_available(),
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=collate_fn,
)

trainer.train()
print("✅ Fine-tuning complete!")



⏳ Fine-tuning Falcon-RW-1B with LoRA...
{'loss': 2.4611, 'grad_norm': 3.154905319213867, 'learning_rate': 0.00018400000000000003, 'epoch': 0.0003020600495378481}
{'loss': 3.5409, 'grad_norm': nan, 'learning_rate': 0.000164, 'epoch': 0.0006041200990756962}
{'loss': 2.8104, 'grad_norm': nan, 'learning_rate': 0.000144, 'epoch': 0.0009061801486135444}
{'loss': 3.0499, 'grad_norm': 5.903543949127197, 'learning_rate': 0.000124, 'epoch': 0.0012082401981513924}
{'loss': 3.0401, 'grad_norm': 5.709779262542725, 'learning_rate': 0.00010400000000000001, 'epoch': 0.0015103002476892407}
{'loss': 3.2153, 'grad_norm': 4.989292621612549, 'learning_rate': 8.4e-05, 'epoch': 0.0018123602972270887}
{'loss': 3.0954, 'grad_norm': 5.696595191955566, 'learning_rate': 6.400000000000001e-05, 'epoch': 0.0021144203467649367}
{'loss': 3.664, 'grad_norm': 4.726617336273193, 'learning_rate': 4.4000000000000006e-05, 'epoch': 0.002416480396302785}
{'loss': 3.0992, 'grad_norm': 3.546311140060425, 'learning_rate': 2.4e-

### **Step 9: Compare model outputs after fine-tuning**

- Generate outputs for the same set of prompts using the fine-tuned model
- These outputs are compared side-by-side with the baseline outputs generated earlier.


Comparing outputs before and after fine-tuning clearly demonstrates the impact of the training process. It shows how the model's responses change after being fine-tuned with PEFT, providing practical insights into the effectiveness of this method.

In [30]:

print("\n=== Comparing Outputs BEFORE and AFTER Fine-Tuning ===")
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output_ids = model.generate(
        **inputs,
        use_cache=False,
        max_new_tokens=64,
        temperature=0.8,
        top_p=0.9,
    )
    after_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print(f"\nPrompt: {prompt}")
    print("| Before Fine-Tuning |")
    print(before_outputs[prompt])
    print("\n| After Fine-Tuning |")
    print(after_text)
    print("=" * 80)



=== Comparing Outputs BEFORE and AFTER Fine-Tuning ===

Prompt: What is artificial intelligence?
| Before Fine-Tuning |
What is artificial intelligence?
Artificial intelligence (AI) is a field of study that aims to create machines that think and act like humans.
Artificial intelligence is a field of study that aims to create machines that think and act like humans.
Artificial intelligence is a field of study that aims to create machines that think and act like

| After Fine-Tuning |
What is artificial intelligence?
Artificial intelligence (AI) is the science and engineering of creating machines that exhibit intelligence.
The term was coined by John McCarthy in 1956 to describe the ability of computers to learn and to exhibit “intelligent behavior”.
The term was first used in the context of the development of the first artificial

Prompt: Explain the concept of quantum computing in simple terms.
| Before Fine-Tuning |
Explain the concept of quantum computing in simple terms.
Quantum co

### __Conclusion__

By following these steps, you have successfully fine-tuned the Falcon 1B model using Parameter-Efficient Fine-Tuning (PEFT) with LoRA. You learned how to configure LoRA parameters such as rank, scaling factor, and dropout to train only the most impactful low-rank layers instead of the entire model. Using a subset of WikiText-2, you efficiently adapted the model with minimal computational overhead.

The comparison of outputs before and after fine-tuning clearly illustrates how PEFT enhances the model’s contextual reasoning and response quality. This approach demonstrates that LoRA provides a cost-effective and scalable strategy for fine-tuning large models while maintaining their strong baseline capabilities.

---